# Notebook 5 — Predicción meteorológica con RNN, LSTM y GRU

**Curso:** Deep Learning  
**Tema:** Predicción de series de tiempo multivariadas  
**Aplicación:** Pronóstico de temperatura a partir de temperatura, presión atmosférica y variables meteorológicas de varias estaciones  
**Framework:** TensorFlow/Keras  
**Docente:** Jersson  


**Estudiantes:**  
- Juan David Tejedor Medina  
- Miguel Guerardo Moreno Aveldaño  

---

Este notebook reemplaza la práctica de clasificación de texto por una aplicación de **pronóstico meteorológico**.

Se trabajará con datos horarios de varias estaciones meteorológicas y se compararán tres arquitecturas:

1. `SimpleRNN`
2. `LSTM`
3. `GRU`

El objetivo será predecir la **temperatura de la siguiente hora** utilizando una ventana histórica de observaciones.


### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


## Objetivos de aprendizaje

Al finalizar esta práctica, el estudiante podrá:

- reconocer una serie de tiempo multivariada;
- interpretar variables meteorológicas horarias;
- incorporar información de diferentes estaciones;
- crear ventanas temporales de entrada y salida;
- realizar una división cronológica sin fuga de información;
- normalizar variables usando únicamente entrenamiento;
- construir modelos `SimpleRNN`, `LSTM` y `GRU`;
- comparar los modelos mediante MAE, MSE, RMSE, \(R^2\) y MAPE;
- comparar los modelos contra una línea base de persistencia;
- visualizar pronósticos y residuos;
- realizar predicciones para cada estación meteorológica.


## Planteamiento del problema

Disponemos de mediciones horarias de varias estaciones:

- temperatura;
- presión atmosférica;
- humedad relativa;
- velocidad del viento;
- precipitación;
- estación meteorológica;
- variables cíclicas de hora y día del año.

Para cada instante \(t\), se utilizarán las últimas 24 horas:

\[
X_t=
[\mathbf{x}_{t-23},\ldots,\mathbf{x}_{t}]
\]

para predecir:

\[
y_{t+1}=\text{temperatura de la siguiente hora}
\]

Por tanto, se trata de una regresión supervisada de series de tiempo.


## 1. Preparación del entorno

In [ ]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Dispositivos:", tf.config.list_physical_devices())


# Parte I — Construcción del conjunto meteorológico

Para que el notebook sea reproducible y no dependa de servicios externos, se generará un conjunto meteorológico sintético con patrones realistas.

Las estaciones representarán tres contextos climáticos:

- estación de montaña;
- estación urbana;
- estación de valle.

Cada una tendrá diferencias en temperatura media, presión y amplitud térmica.


## 2. Parámetros de simulación

In [ ]:
N_DAYS = 365
FREQUENCY = "h"
START_DATE = "2024-01-01"
WINDOW_SIZE = 6
FORECAST_HORIZON = 1

stations = {
    "Montaña": {
        "temperature_offset": -4.0,
        "pressure_base": 780.0,
        "daily_amplitude": 6.0,
        "humidity_offset": 8.0,
    },
    "Urbana": {
        "temperature_offset": 2.0,
        "pressure_base": 850.0,
        "daily_amplitude": 5.0,
        "humidity_offset": -4.0,
    },
    "Valle": {
        "temperature_offset": 0.0,
        "pressure_base": 820.0,
        "daily_amplitude": 7.0,
        "humidity_offset": 3.0,
    },
}

print("Días:", N_DAYS)
print("Ventana histórica:", WINDOW_SIZE, "horas")
print("Horizonte:", FORECAST_HORIZON, "hora")
print("Estaciones:", list(stations))


## 3. Función de generación

La temperatura combinará:

- ciclo diario;
- ciclo estacional;
- efecto de la presión;
- humedad;
- variación propia de cada estación;
- ruido aleatorio.

Estos datos son didácticos y no deben utilizarse para decisiones meteorológicas reales.


In [ ]:
def generate_station_weather(
    station_name,
    config,
    start_date,
    n_days,
    seed,
):
    rng = np.random.default_rng(seed)

    n_hours = n_days * 24
    timestamps = pd.date_range(
        start=start_date,
        periods=n_hours,
        freq=FREQUENCY,
    )

    hour = timestamps.hour.to_numpy()
    day_of_year = timestamps.dayofyear.to_numpy()

    daily_cycle = np.sin(
        2.0 * np.pi * (hour - 6) / 24.0
    )
    annual_cycle = np.sin(
        2.0 * np.pi * (day_of_year - 80) / 365.25
    )

    synoptic_cycle = np.sin(
        2.0 * np.pi * np.arange(n_hours) / (24.0 * 7.0)
    )

    pressure = (
        config["pressure_base"]
        + 5.0 * synoptic_cycle
        + 2.0 * np.cos(2.0 * np.pi * hour / 24.0)
        + rng.normal(0.0, 1.3, n_hours)
    )

    humidity = (
        68.0
        - 18.0 * daily_cycle
        - 8.0 * annual_cycle
        + config["humidity_offset"]
        + rng.normal(0.0, 4.0, n_hours)
    )
    humidity = np.clip(humidity, 20.0, 100.0)

    wind_speed = (
        2.5
        + 1.2 * np.maximum(daily_cycle, 0)
        + 0.8 * np.abs(synoptic_cycle)
        + rng.gamma(shape=1.5, scale=0.5, size=n_hours)
    )

    rain_probability = 1.0 / (
        1.0 + np.exp(
            -(
                0.10 * (humidity - 75.0)
                - 0.25 * (pressure - config["pressure_base"])
            )
        )
    )

    rain_event = rng.binomial(1, np.clip(rain_probability, 0, 1))
    precipitation = rain_event * rng.gamma(
        shape=1.2,
        scale=1.4,
        size=n_hours,
    )

    temperature = (
        16.0
        + config["temperature_offset"]
        + config["daily_amplitude"] * daily_cycle
        + 4.0 * annual_cycle
        - 0.08 * (humidity - 60.0)
        + 0.10 * (pressure - config["pressure_base"])
        - 0.25 * precipitation
        + rng.normal(0.0, 0.8, n_hours)
    )

    return pd.DataFrame({
        "timestamp": timestamps,
        "station": station_name,
        "temperature": temperature,
        "pressure": pressure,
        "humidity": humidity,
        "wind_speed": wind_speed,
        "precipitation": precipitation,
    })


In [ ]:
weather_frames = []

for index, (station_name, config) in enumerate(stations.items()):
    station_df = generate_station_weather(
        station_name=station_name,
        config=config,
        start_date=START_DATE,
        n_days=N_DAYS,
        seed=SEED + index,
    )
    weather_frames.append(station_df)

weather_df = pd.concat(
    weather_frames,
    ignore_index=True,
)

weather_df.head()


In [ ]:
print("Forma del conjunto:", weather_df.shape)
print("\nRegistros por estación:")
print(weather_df["station"].value_counts())
print("\nRango temporal:")
print(weather_df["timestamp"].min(), "a", weather_df["timestamp"].max())


## 4. Estadística descriptiva por estación

In [ ]:
weather_df.groupby("station")[
    ["temperature", "pressure", "humidity", "wind_speed", "precipitation"]
].agg(["mean", "std", "min", "max"]).round(2)


## 5. Visualización de temperatura

In [ ]:
sample_days = 14
sample_hours = sample_days * 24

plt.figure(figsize=(14, 6))

for station_name in stations:
    station_sample = (
        weather_df[weather_df["station"] == station_name]
        .sort_values("timestamp")
        .head(sample_hours)
    )

    plt.plot(
        station_sample["timestamp"],
        station_sample["temperature"],
        label=station_name,
    )

plt.xlabel("Fecha")
plt.ylabel("Temperatura")
plt.title(f"Temperatura horaria durante {sample_days} días")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Relación entre presión y temperatura

In [ ]:
plt.figure(figsize=(9, 6))

for station_name in stations:
    station_sample = weather_df[
        weather_df["station"] == station_name
    ].sample(700, random_state=SEED)

    plt.scatter(
        station_sample["pressure"],
        station_sample["temperature"],
        alpha=0.35,
        label=station_name,
    )

plt.xlabel("Presión atmosférica")
plt.ylabel("Temperatura")
plt.title("Presión frente a temperatura")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Parte II — Ingeniería de características temporales

La hora del día y el día del año son variables cíclicas.

No conviene usar directamente:

\[
0,1,\ldots,23
\]

porque las horas 23 y 0 son cercanas, aunque numéricamente parezcan distantes.

Se utilizarán transformaciones seno y coseno.


## 7. Variables cíclicas

In [ ]:
weather_df = weather_df.sort_values(
    ["station", "timestamp"]
).reset_index(drop=True)

weather_df["hour"] = weather_df["timestamp"].dt.hour
weather_df["day_of_year"] = weather_df["timestamp"].dt.dayofyear

weather_df["hour_sin"] = np.sin(
    2.0 * np.pi * weather_df["hour"] / 24.0
)
weather_df["hour_cos"] = np.cos(
    2.0 * np.pi * weather_df["hour"] / 24.0
)

weather_df["day_sin"] = np.sin(
    2.0 * np.pi * weather_df["day_of_year"] / 365.25
)
weather_df["day_cos"] = np.cos(
    2.0 * np.pi * weather_df["day_of_year"] / 365.25
)

weather_df.head()


## 8. Codificación de estaciones

Las estaciones se codificarán mediante variables binarias.


In [ ]:
station_dummies = pd.get_dummies(
    weather_df["station"],
    prefix="station",
    dtype=float,
)

weather_model_df = pd.concat(
    [weather_df, station_dummies],
    axis=1,
)

station_columns = station_dummies.columns.tolist()

print("Columnas de estación:", station_columns)


## 9. Variables de entrada

Las características serán:

- temperatura histórica;
- presión;
- humedad;
- viento;
- precipitación;
- hora cíclica;
- día del año cíclico;
- estación meteorológica.

La variable objetivo será la temperatura de la siguiente hora.


In [ ]:
feature_columns = [
    "temperature",
    "pressure",
    "humidity",
    "wind_speed",
    "precipitation",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
] + station_columns

target_column = "temperature"

print("Número de características:", len(feature_columns))
print(feature_columns)


# Parte III — División cronológica

En series de tiempo no se debe mezclar aleatoriamente el pasado y el futuro.

Para cada estación se utilizará:

- 70 % inicial: entrenamiento;
- 15 % siguiente: validación;
- 15 % final: prueba.

Esta división simula un escenario real de pronóstico.


## 10. Índices temporales por estación

In [ ]:
split_summary = []

for station_name in stations:
    station_count = (
        weather_model_df["station"] == station_name
    ).sum()

    train_end = int(station_count * 0.70)
    val_end = int(station_count * 0.85)

    split_summary.append({
        "station": station_name,
        "total": station_count,
        "train": train_end,
        "validation": val_end - train_end,
        "test": station_count - val_end,
    })

pd.DataFrame(split_summary)


## 11. Ajuste de escaladores

Los escaladores se ajustan únicamente con observaciones del período de entrenamiento.


In [ ]:
train_rows = []
val_rows = []
test_rows = []

for station_name in stations:
    station_data = (
        weather_model_df[
            weather_model_df["station"] == station_name
        ]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    n = len(station_data)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_rows.append(station_data.iloc[:train_end])
    val_rows.append(station_data.iloc[train_end:val_end])
    test_rows.append(station_data.iloc[val_end:])

train_df = pd.concat(train_rows, ignore_index=True)
val_df = pd.concat(val_rows, ignore_index=True)
test_df = pd.concat(test_rows, ignore_index=True)

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

feature_scaler.fit(train_df[feature_columns])
target_scaler.fit(train_df[[target_column]])

print("Escaladores ajustados exclusivamente con entrenamiento.")


# Parte IV — Creación de ventanas

Cada ejemplo tendrá forma:

\[
(\text{24 pasos},\text{número de características})
\]

La salida será un escalar: temperatura de la siguiente hora.


## 12. Función para crear secuencias por estación

In [ ]:
def create_station_sequences(
    dataframe,
    feature_columns,
    target_column,
    feature_scaler,
    target_scaler,
    window_size,
    horizon=1,
):
    X_sequences = []
    y_targets = []
    metadata = []

    for station_name, station_data in dataframe.groupby("station"):
        station_data = station_data.sort_values(
            "timestamp"
        ).reset_index(drop=True)

        features_scaled = feature_scaler.transform(
            station_data[feature_columns]
        )
        target_scaled = target_scaler.transform(
            station_data[[target_column]]
        ).ravel()

        max_start = (
            len(station_data)
            - window_size
            - horizon
            + 1
        )

        for start in range(max_start):
            end = start + window_size
            target_index = end + horizon - 1

            X_sequences.append(
                features_scaled[start:end]
            )
            y_targets.append(
                target_scaled[target_index]
            )

            metadata.append({
                "station": station_name,
                "target_timestamp": station_data.loc[
                    target_index,
                    "timestamp",
                ],
                "target_temperature": station_data.loc[
                    target_index,
                    target_column,
                ],
                "last_observed_temperature": station_data.loc[
                    end - 1,
                    target_column,
                ],
            })

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_targets, dtype=np.float32).reshape(-1, 1),
        pd.DataFrame(metadata),
    )


In [ ]:
X_train, y_train, meta_train = create_station_sequences(
    train_df,
    feature_columns,
    target_column,
    feature_scaler,
    target_scaler,
    WINDOW_SIZE,
    FORECAST_HORIZON,
)

X_val, y_val, meta_val = create_station_sequences(
    val_df,
    feature_columns,
    target_column,
    feature_scaler,
    target_scaler,
    WINDOW_SIZE,
    FORECAST_HORIZON,
)

X_test, y_test, meta_test = create_station_sequences(
    test_df,
    feature_columns,
    target_column,
    feature_scaler,
    target_scaler,
    WINDOW_SIZE,
    FORECAST_HORIZON,
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


## 13. Interpretación de dimensiones

Si la salida es, por ejemplo:

```text
X_train: (N, 24, 12)
```

significa:

- \(N\): número de ventanas;
- 24: horas históricas;
- 12: variables por cada hora.


## 14. Visualización de una ventana

In [ ]:
sample_index = 0

sample_station = meta_train.loc[sample_index, "station"]
sample_target_time = meta_train.loc[
    sample_index,
    "target_timestamp",
]

temperature_feature_index = feature_columns.index(
    "temperature"
)

sample_temperature_scaled = X_train[
    sample_index,
    :,
    temperature_feature_index,
].reshape(-1, 1)

# Reconstrucción aproximada de la temperatura usando
# la media y desviación del escalador de características.
temperature_mean = feature_scaler.mean_[
    temperature_feature_index
]
temperature_scale = feature_scaler.scale_[
    temperature_feature_index
]

sample_temperature = (
    sample_temperature_scaled.ravel()
    * temperature_scale
    + temperature_mean
)

plt.figure(figsize=(10, 5))
plt.plot(
    range(-WINDOW_SIZE, 0),
    sample_temperature,
    marker="o",
)
plt.axvline(-1, linestyle="--")
plt.xlabel("Horas respecto al instante de predicción")
plt.ylabel("Temperatura")
plt.title(
    f"Ventana de entrada — {sample_station}\n"
    f"Objetivo: {sample_target_time}"
)
plt.grid(alpha=0.25)
plt.show()


# Parte V — Línea base de persistencia

Una referencia sencilla consiste en asumir:

\[
\hat y_{t+1}=y_t
\]

Es decir, la temperatura de la siguiente hora será igual a la última observada.

Los modelos recurrentes deben superar esta línea base para justificar su complejidad.


In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(
        y_true,
        y_pred,
    ) * 100.0

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_percent": mape,
    }


y_test_real = meta_test[
    "target_temperature"
].to_numpy()

persistence_predictions = meta_test[
    "last_observed_temperature"
].to_numpy()

baseline_metrics = regression_metrics(
    y_test_real,
    persistence_predictions,
)

pd.DataFrame([{
    "model": "Persistencia",
    **baseline_metrics,
}])


# Parte VI — Funciones auxiliares para modelos recurrentes


In [ ]:
BATCH_SIZE = 128
EPOCHS = 40
RECURRENT_UNITS = 64


def build_recurrent_model(
    recurrent_type,
    input_shape,
    units=64,
):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    if recurrent_type == "SimpleRNN":
        recurrent_layer = tf.keras.layers.SimpleRNN(
            units,
            name="simple_rnn",
        )
    elif recurrent_type == "LSTM":
        recurrent_layer = tf.keras.layers.LSTM(
            units,
            name="lstm",
        )
    elif recurrent_type == "GRU":
        recurrent_layer = tf.keras.layers.GRU(
            units,
            name="gru",
        )
    else:
        raise ValueError(
            "recurrent_type debe ser SimpleRNN, LSTM o GRU"
        )

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=input_shape,
            name="weather_window",
        ),
        recurrent_layer,
        tf.keras.layers.Dense(
            32,
            activation="relu",
            name="dense_hidden",
        ),
        tf.keras.layers.Dropout(
            0.20,
            name="dropout",
        ),
        tf.keras.layers.Dense(
            1,
            activation="linear",
            name="temperature_output",
        ),
    ], name=f"weather_{recurrent_type.lower()}")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            ),
        ],
    )

    return model


def make_callbacks(model_name):
    checkpoint_dir = Path(
        "/content/weather_checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(
                checkpoint_dir / f"{model_name}.keras"
            ),
            monitor="val_loss",
            save_best_only=True,
        ),
    ]


def train_recurrent_model(model, model_name):
    start_time = time.perf_counter()

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=make_callbacks(model_name),
        verbose=1,
    )

    elapsed = time.perf_counter() - start_time

    return history, elapsed


def inverse_target(values_scaled):
    return target_scaler.inverse_transform(
        np.asarray(values_scaled).reshape(-1, 1)
    ).ravel()


def evaluate_recurrent_model(
    model,
    model_name,
    elapsed_time,
):
    predictions_scaled = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    predictions_real = inverse_target(
        predictions_scaled
    )

    metrics = regression_metrics(
        y_test_real,
        predictions_real,
    )

    result = {
        "model": model_name,
        "parameters": model.count_params(),
        "training_time_seconds": elapsed_time,
        **metrics,
    }

    return result, predictions_real


## 15. Función para graficar curvas

In [ ]:
def plot_training_history(history, title):
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(
        history_df["loss"],
        label="Entrenamiento",
    )
    plt.plot(
        history_df["val_loss"],
        label="Validación",
    )
    plt.xlabel("Época")
    plt.ylabel("MSE escalado")
    plt.title(f"{title}: pérdida")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        history_df["mae"],
        label="MAE entrenamiento",
    )
    plt.plot(
        history_df["val_mae"],
        label="MAE validación",
    )
    plt.xlabel("Época")
    plt.ylabel("MAE escalado")
    plt.title(f"{title}: MAE")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


# Parte VII — SimpleRNN


## 16. Construcción de la SimpleRNN

In [ ]:
simple_rnn_model = build_recurrent_model(
    recurrent_type="SimpleRNN",
    input_shape=X_train.shape[1:],
    units=RECURRENT_UNITS,
)

simple_rnn_model.summary()


## 17. Entrenamiento de la SimpleRNN

In [ ]:
history_rnn, time_rnn = train_recurrent_model(
    simple_rnn_model,
    model_name="simple_rnn_weather",
)

print(f"Tiempo: {time_rnn:.2f} segundos")


In [ ]:
plot_training_history(
    history_rnn,
    title="SimpleRNN",
)


## 18. Evaluación de la SimpleRNN

In [ ]:
result_rnn, predictions_rnn = (
    evaluate_recurrent_model(
        simple_rnn_model,
        model_name="SimpleRNN",
        elapsed_time=time_rnn,
    )
)

pd.DataFrame([result_rnn])


# Parte VIII — LSTM


## 19. Construcción de la LSTM

In [ ]:
lstm_model = build_recurrent_model(
    recurrent_type="LSTM",
    input_shape=X_train.shape[1:],
    units=RECURRENT_UNITS,
)

lstm_model.summary()


## 20. Entrenamiento de la LSTM

In [ ]:
history_lstm, time_lstm = train_recurrent_model(
    lstm_model,
    model_name="lstm_weather",
)

print(f"Tiempo: {time_lstm:.2f} segundos")


In [ ]:
plot_training_history(
    history_lstm,
    title="LSTM",
)


## 21. Evaluación de la LSTM

In [ ]:
result_lstm, predictions_lstm = (
    evaluate_recurrent_model(
        lstm_model,
        model_name="LSTM",
        elapsed_time=time_lstm,
    )
)

pd.DataFrame([result_lstm])


# Parte IX — GRU


## 22. Construcción de la GRU

In [ ]:
gru_model = build_recurrent_model(
    recurrent_type="GRU",
    input_shape=X_train.shape[1:],
    units=RECURRENT_UNITS,
)

gru_model.summary()


## 23. Entrenamiento de la GRU

In [ ]:
history_gru, time_gru = train_recurrent_model(
    gru_model,
    model_name="gru_weather",
)

print(f"Tiempo: {time_gru:.2f} segundos")


In [ ]:
plot_training_history(
    history_gru,
    title="GRU",
)


## 24. Evaluación de la GRU

In [ ]:
result_gru, predictions_gru = (
    evaluate_recurrent_model(
        gru_model,
        model_name="GRU",
        elapsed_time=time_gru,
    )
)

pd.DataFrame([result_gru])


# Parte X — Comparación de modelos


## 25. Tabla general

In [ ]:
results_df = pd.DataFrame([
    {
        "model": "Persistencia",
        "parameters": 0,
        "training_time_seconds": 0.0,
        **baseline_metrics,
    },
    result_rnn,
    result_lstm,
    result_gru,
])

results_df


## 26. Comparación de RMSE

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    results_df["model"],
    results_df["RMSE"],
)
plt.ylabel("RMSE de temperatura")
plt.title("Comparación de RMSE")
plt.grid(axis="y", alpha=0.25)
plt.show()


## 27. Comparación de MAE

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    results_df["model"],
    results_df["MAE"],
)
plt.ylabel("MAE de temperatura")
plt.title("Comparación de MAE")
plt.grid(axis="y", alpha=0.25)
plt.show()


## 28. Comparación de tiempo y parámetros

In [ ]:
model_results = results_df[
    results_df["model"] != "Persistencia"
]

plt.figure(figsize=(8, 5))
plt.bar(
    model_results["model"],
    model_results["training_time_seconds"],
)
plt.ylabel("Tiempo de entrenamiento (s)")
plt.title("Costo temporal")
plt.grid(axis="y", alpha=0.25)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(
    model_results["model"],
    model_results["parameters"],
)
plt.ylabel("Parámetros")
plt.title("Complejidad de los modelos")
plt.grid(axis="y", alpha=0.25)
plt.show()


# Parte XI — Análisis temporal de predicciones


## 29. Selección automática del mejor modelo

In [ ]:
prediction_map = {
    "Persistencia": persistence_predictions,
    "SimpleRNN": predictions_rnn,
    "LSTM": predictions_lstm,
    "GRU": predictions_gru,
}

best_model_name = (
    results_df
    .sort_values("RMSE")
    .iloc[0]["model"]
)

best_predictions = prediction_map[
    best_model_name
]

print("Mejor modelo según RMSE:", best_model_name)


## 30. Pronóstico para una estación

In [ ]:
station_to_plot = "Montaña"
plot_hours = 7 * 24

station_mask = (
    meta_test["station"] == station_to_plot
)
station_indices = np.where(station_mask)[0][:plot_hours]

station_times = meta_test.loc[
    station_indices,
    "target_timestamp",
]
station_real = y_test_real[station_indices]
station_pred = best_predictions[station_indices]

plt.figure(figsize=(14, 6))
plt.plot(
    station_times,
    station_real,
    label="Temperatura real",
)
plt.plot(
    station_times,
    station_pred,
    label=f"Predicción: {best_model_name}",
)
plt.xlabel("Fecha")
plt.ylabel("Temperatura")
plt.title(
    f"Pronóstico horario — Estación {station_to_plot}"
)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 31. Valores reales frente a predichos

In [ ]:
min_value = min(
    y_test_real.min(),
    best_predictions.min(),
)
max_value = max(
    y_test_real.max(),
    best_predictions.max(),
)

plt.figure(figsize=(7, 7))
plt.scatter(
    y_test_real,
    best_predictions,
    alpha=0.35,
)
plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
)
plt.xlabel("Temperatura real")
plt.ylabel("Temperatura predicha")
plt.title(
    f"Real frente a predicho — {best_model_name}"
)
plt.grid(alpha=0.25)
plt.show()


# Parte XII — Análisis de residuos


## 32. Residuos globales

In [ ]:
residuals = (
    y_test_real
    - best_predictions
)

plt.figure(figsize=(9, 5))
plt.scatter(
    best_predictions,
    residuals,
    alpha=0.35,
)
plt.axhline(0, linestyle="--")
plt.xlabel("Temperatura predicha")
plt.ylabel("Residuo")
plt.title(
    f"Residuos — {best_model_name}"
)
plt.grid(alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    residuals,
    bins=35,
    edgecolor="black",
)
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.title("Distribución de residuos")
plt.grid(alpha=0.20)
plt.show()


## 33. Desempeño por estación

In [ ]:
station_results = []

for station_name in stations:
    mask = (
        meta_test["station"].to_numpy()
        == station_name
    )

    metrics = regression_metrics(
        y_test_real[mask],
        best_predictions[mask],
    )

    station_results.append({
        "station": station_name,
        "model": best_model_name,
        **metrics,
    })

station_results_df = pd.DataFrame(
    station_results
)

station_results_df


## 34. RMSE por estación

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    station_results_df["station"],
    station_results_df["RMSE"],
)
plt.ylabel("RMSE")
plt.title(
    f"RMSE por estación — {best_model_name}"
)
plt.grid(axis="y", alpha=0.25)
plt.show()


# Parte XIII — Pronóstico de una nueva ventana


## 35. Función de predicción

In [ ]:
def forecast_next_temperature(
    model,
    station_dataframe,
    feature_columns,
    feature_scaler,
    target_scaler,
    window_size,
):
    station_dataframe = station_dataframe.sort_values(
        "timestamp"
    )

    if len(station_dataframe) < window_size:
        raise ValueError(
            "No existen suficientes observaciones."
        )

    last_window = station_dataframe[
        feature_columns
    ].tail(window_size)

    last_window_scaled = feature_scaler.transform(
        last_window
    ).astype(np.float32)

    input_tensor = last_window_scaled[
        np.newaxis,
        ...,
    ]

    prediction_scaled = model.predict(
        input_tensor,
        verbose=0,
    )

    prediction = target_scaler.inverse_transform(
        prediction_scaled.reshape(-1, 1)
    ).ravel()[0]

    return prediction


In [ ]:
best_neural_models = {
    "SimpleRNN": simple_rnn_model,
    "LSTM": lstm_model,
    "GRU": gru_model,
}

if best_model_name == "Persistencia":
    print(
        "La línea base fue la mejor. Para esta demostración "
        "se utilizará el mejor modelo neuronal."
    )
    best_neural_name = (
        results_df[
            results_df["model"] != "Persistencia"
        ]
        .sort_values("RMSE")
        .iloc[0]["model"]
    )
else:
    best_neural_name = best_model_name

best_neural_model = best_neural_models[
    best_neural_name
]

station_name = "Valle"
station_full_data = weather_model_df[
    weather_model_df["station"] == station_name
]

next_temperature = forecast_next_temperature(
    model=best_neural_model,
    station_dataframe=station_full_data,
    feature_columns=feature_columns,
    feature_scaler=feature_scaler,
    target_scaler=target_scaler,
    window_size=WINDOW_SIZE,
)

print("Estación:", station_name)
print(
    "Temperatura pronosticada para la siguiente hora:",
    f"{next_temperature:.2f}"
)


# Parte XIV — Actividades experimentales desarrolladas

En esta sección se resuelven los cuatro experimentos solicitados. Para que la comparación sea justa se usa una **GRU de una capa como configuración de control** y se modifica una sola condición dentro de cada experimento.

Configuración común:

- semilla fija para reproducibilidad;
- división cronológica 70 % / 15 % / 15 % por estación;
- escaladores ajustados exclusivamente con entrenamiento;
- optimizador Adam y pérdida MSE;
- máximo de 15 épocas con parada temprana;
- evaluación en grados de temperatura mediante MAE, RMSE y \(R^2\).

La configuración `ventana=24`, `horizonte=1`, `todas las variables` y `GRU de una capa` aparece en varios experimentos y se entrena una sola vez; después se reutiliza exactamente el mismo resultado.

## 36. Preparación común para los experimentos

Las funciones siguientes generan las ventanas según cada configuración, construyen la arquitectura indicada y registran arquitectura, parámetros, épocas, tiempo y métricas de prueba. Para mantener un tiempo razonable en Colab se emplea una GRU compacta de 16 unidades; lo importante es conservar la misma capacidad cuando se compara ventana, horizonte o variables.

In [ ]:
import gc
import hashlib

EXPERIMENT_EPOCHS = 15
EXPERIMENT_BATCH_SIZE = 512
EXPERIMENT_UNITS = 16

feature_sets = {
    "Solo temperatura": ["temperature"],
    "Temperatura + presión": ["temperature", "pressure"],
    "Todas las variables": feature_columns,
    "Todas sin estación": [c for c in feature_columns if c not in station_columns],
}

experiment_cache = {}
experiment_rows = []


def stable_seed(*parts):
    text = "|".join(map(str, parts)).encode("utf-8")
    return SEED + int(hashlib.md5(text).hexdigest()[:6], 16) % 10000


def prepare_experiment_data(window, horizon, selected_features):
    local_feature_scaler = StandardScaler().fit(train_df[selected_features])
    local_target_scaler = StandardScaler().fit(train_df[[target_column]])

    train_pack = create_station_sequences(
        train_df, selected_features, target_column,
        local_feature_scaler, local_target_scaler, window, horizon
    )
    val_pack = create_station_sequences(
        val_df, selected_features, target_column,
        local_feature_scaler, local_target_scaler, window, horizon
    )
    test_pack = create_station_sequences(
        test_df, selected_features, target_column,
        local_feature_scaler, local_target_scaler, window, horizon
    )
    return train_pack, val_pack, test_pack, local_target_scaler


def build_experiment_model(input_shape, architecture, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    inputs = tf.keras.Input(shape=input_shape, name="ventana_meteorologica")

    if architecture == "GRU: una capa":
        x = tf.keras.layers.GRU(EXPERIMENT_UNITS)(inputs)
    elif architecture == "GRU: dos capas":
        x = tf.keras.layers.GRU(EXPERIMENT_UNITS, return_sequences=True)(inputs)
        x = tf.keras.layers.GRU(EXPERIMENT_UNITS)(x)
    elif architecture == "GRU bidireccional":
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(EXPERIMENT_UNITS)
        )(inputs)
    elif architecture == "CNN 1D + GRU":
        x = tf.keras.layers.Conv1D(
            filters=16, kernel_size=3, padding="same", activation="relu"
        )(inputs)
        x = tf.keras.layers.GRU(EXPERIMENT_UNITS)(x)
    else:
        raise ValueError(f"Arquitectura desconocida: {architecture}")

    x = tf.keras.layers.Dense(8, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="linear")(x)
    model = tf.keras.Model(inputs, outputs, name="modelo_experimental")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


def run_experiment(experiment, variant, window, horizon, feature_label,
                   architecture="GRU: una capa"):
    selected_features = feature_sets[feature_label]
    key = (window, horizon, tuple(selected_features), architecture)

    if key not in experiment_cache:
        seed = stable_seed(window, horizon, feature_label, architecture)
        (Xtr, ytr, _), (Xv, yv, _), (Xte, yte, meta_te), y_scaler = (
            prepare_experiment_data(window, horizon, selected_features)
        )
        model = build_experiment_model(Xtr.shape[1:], architecture, seed)
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=3, restore_best_weights=True
            )
        ]
        start = time.perf_counter()
        history = model.fit(
            Xtr, ytr,
            validation_data=(Xv, yv),
            epochs=EXPERIMENT_EPOCHS,
            batch_size=EXPERIMENT_BATCH_SIZE,
            callbacks=callbacks,
            shuffle=False,
            verbose=0,
        )
        elapsed = time.perf_counter() - start

        pred_scaled = model.predict(
            Xte, batch_size=EXPERIMENT_BATCH_SIZE, verbose=0
        )
        pred_real = y_scaler.inverse_transform(pred_scaled).ravel()
        y_real = meta_te["target_temperature"].to_numpy()
        metrics = regression_metrics(y_real, pred_real)
        persistence_rmse = regression_metrics(
            y_real, meta_te["last_observed_temperature"].to_numpy()
        )["RMSE"]

        experiment_cache[key] = {
            "architecture": architecture,
            "parameters": model.count_params(),
            "epochs": len(history.history["loss"]),
            "training_time_seconds": elapsed,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "persistence_RMSE": persistence_rmse,
            "predictions": pred_real,
            "y_real": y_real,
            "metadata": meta_te,
        }
        del model, Xtr, ytr, Xv, yv, Xte, yte
        gc.collect()

    cached = experiment_cache[key]
    row = {
        "experiment": experiment,
        "variant": variant,
        "architecture": cached["architecture"],
        "window": window,
        "horizon": horizon,
        "features": feature_label,
        "n_features": len(selected_features),
        "parameters": cached["parameters"],
        "epochs": cached["epochs"],
        "training_time_seconds": cached["training_time_seconds"],
        "MAE": cached["MAE"],
        "RMSE": cached["RMSE"],
        "R2": cached["R2"],
        "persistence_RMSE": cached["persistence_RMSE"],
    }
    experiment_rows.append(row)
    print(
        f"{experiment} | {variant}: RMSE={row['RMSE']:.4f}, "
        f"MAE={row['MAE']:.4f}, R²={row['R2']:.4f}, "
        f"épocas={row['epochs']}"
    )
    return row


def experiment_table(letter):
    columns = [
        "variant", "architecture", "window", "horizon", "features",
        "n_features", "parameters", "epochs", "training_time_seconds",
        "MAE", "RMSE", "R2", "persistence_RMSE"
    ]
    return pd.DataFrame(
        [r for r in experiment_rows if r["experiment"] == letter]
    )[columns]


print("Funciones experimentales listas.")
print("Máximo de épocas:", EXPERIMENT_EPOCHS)
print("Tamaño de lote:", EXPERIMENT_BATCH_SIZE)

## 37. Experimento A — Tamaño de ventana

Se mantienen fijos el horizonte de 1 hora, todas las variables y la GRU de una capa. Solo cambia la cantidad de horas históricas: 6, 12, 24, 48 y 72.

In [ ]:
for window in [6, 12, 24, 48, 72]:
    run_experiment(
        experiment="A",
        variant=f"{window} horas",
        window=window,
        horizon=1,
        feature_label="Todas las variables",
    )

results_a = experiment_table("A")
print("\nTabla del experimento A")
print(results_a.round(4).to_string(index=False))

best_a = results_a.loc[results_a["RMSE"].idxmin()]
print(
    f"\nConclusión A: la ventana con menor RMSE fue {best_a['variant']} "
    f"(RMSE={best_a['RMSE']:.4f}). Una ventana más larga no garantiza "
    "una mejora: incorpora más contexto, pero también aumenta el costo y puede "
    "añadir información poco relevante."
)

## 38. Experimento B — Horizonte de pronóstico

Se fija una ventana de 24 horas, todas las variables y la misma GRU. Solo cambia cuántas horas hacia el futuro se desea predecir: 1, 3, 6 y 24.

In [ ]:
for horizon in [1, 3, 6, 24]:
    run_experiment(
        experiment="B",
        variant=f"{horizon} hora(s) adelante",
        window=24,
        horizon=horizon,
        feature_label="Todas las variables",
    )

results_b = experiment_table("B")
print("\nTabla del experimento B")
print(results_b.round(4).to_string(index=False))

best_b = results_b.loc[results_b["RMSE"].idxmin()]
worst_b = results_b.loc[results_b["RMSE"].idxmax()]
print(
    f"\nConclusión B: el menor error apareció a {best_b['variant']} "
    f"(RMSE={best_b['RMSE']:.4f}) y el mayor a {worst_b['variant']} "
    f"(RMSE={worst_b['RMSE']:.4f}). Al crecer el horizonte aumenta la "
    "incertidumbre porque el objetivo está más alejado de las observaciones."
)

## 39. Experimento C — Variables de entrada

Se mantienen una ventana de 24 horas, horizonte de 1 hora y una GRU de una capa. Se comparan cuatro grupos de características para estimar cuánto aporta la información adicional.

In [ ]:
for feature_label in [
    "Solo temperatura",
    "Temperatura + presión",
    "Todas las variables",
    "Todas sin estación",
]:
    run_experiment(
        experiment="C",
        variant=feature_label,
        window=24,
        horizon=1,
        feature_label=feature_label,
    )

results_c = experiment_table("C")
print("\nTabla del experimento C")
print(results_c.round(4).to_string(index=False))

best_c = results_c.loc[results_c["RMSE"].idxmin()]
station_row = results_c[results_c["variant"] == "Todas las variables"].iloc[0]
no_station_row = results_c[results_c["variant"] == "Todas sin estación"].iloc[0]
station_effect = no_station_row["RMSE"] - station_row["RMSE"]
print(
    f"\nConclusión C: el grupo con menor RMSE fue '{best_c['variant']}' "
    f"(RMSE={best_c['RMSE']:.4f}). Al retirar la identificación de estación, "
    f"el RMSE cambió en {station_effect:+.4f}; esa comparación cuantifica la "
    "información climática propia de cada ubicación."
)

## 40. Experimento D — Arquitectura

Se fijan la ventana de 24 horas, el horizonte de 1 hora, todas las variables y el mismo número base de unidades. Solo cambia la arquitectura: una GRU, dos GRU apiladas, una GRU bidireccional y una CNN 1D seguida de GRU.

In [ ]:
for architecture in [
    "GRU: una capa",
    "GRU: dos capas",
    "GRU bidireccional",
    "CNN 1D + GRU",
]:
    run_experiment(
        experiment="D",
        variant=architecture,
        window=24,
        horizon=1,
        feature_label="Todas las variables",
        architecture=architecture,
    )

results_d = experiment_table("D")
print("\nTabla del experimento D")
print(results_d.round(4).to_string(index=False))

best_d = results_d.loc[results_d["RMSE"].idxmin()]
fastest_d = results_d.loc[results_d["training_time_seconds"].idxmin()]
print(
    f"\nConclusión D: '{best_d['variant']}' obtuvo el menor RMSE "
    f"({best_d['RMSE']:.4f}) con {int(best_d['parameters'])} parámetros. "
    f"La arquitectura más rápida fue '{fastest_d['variant']}'. El mejor modelo "
    "no se elige solo por error: también deben considerarse parámetros y tiempo."
)

## 41. Tabla consolidada y visualización

La tabla siguiente reúne las 17 comparaciones solicitadas. La línea de persistencia se calcula para cada combinación de ventana y horizonte, por lo que sirve como referencia directa del valor agregado por la red.

In [ ]:
experiment_results_df = pd.DataFrame(experiment_rows)

ordered_columns = [
    "experiment", "variant", "architecture", "window", "horizon",
    "features", "n_features", "parameters", "epochs",
    "training_time_seconds", "MAE", "RMSE", "R2", "persistence_RMSE"
]
experiment_results_df = experiment_results_df[ordered_columns]

print("RESULTADOS CONSOLIDADOS")
print(experiment_results_df.round(4).to_string(index=False))

best_overall = experiment_results_df.loc[
    experiment_results_df["RMSE"].idxmin()
]
print(
    "\nMejor resultado global: "
    f"experimento {best_overall['experiment']}, {best_overall['variant']}, "
    f"RMSE={best_overall['RMSE']:.4f}, MAE={best_overall['MAE']:.4f}, "
    f"R²={best_overall['R2']:.4f}."
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (letter, title) in zip(
    axes.ravel(),
    [
        ("A", "A — Tamaño de ventana"),
        ("B", "B — Horizonte"),
        ("C", "C — Variables de entrada"),
        ("D", "D — Arquitectura"),
    ],
):
    subset = experiment_results_df[experiment_results_df["experiment"] == letter]
    ax.bar(subset["variant"], subset["RMSE"], color="#3B82F6")
    ax.set_title(title)
    ax.set_ylabel("RMSE")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Comparación de los cuatro experimentos", fontsize=16)
fig.tight_layout()
plt.show()

In [ ]:
best_key = (
    int(best_overall["window"]),
    int(best_overall["horizon"]),
    tuple(feature_sets[best_overall["features"]]),
    best_overall["architecture"],
)
best_pack = experiment_cache[best_key]
best_meta = best_pack["metadata"]
best_y = best_pack["y_real"]
best_pred = best_pack["predictions"]

station_to_show = "Montaña"
indices = np.where(best_meta["station"].to_numpy() == station_to_show)[0][:168]

plt.figure(figsize=(14, 5))
plt.plot(
    best_meta.loc[indices, "target_timestamp"],
    best_y[indices],
    label="Temperatura real",
)
plt.plot(
    best_meta.loc[indices, "target_timestamp"],
    best_pred[indices],
    label="Predicción",
)
plt.title(
    f"Mejor configuración global — estación {station_to_show}\n"
    f"{best_overall['variant']} | RMSE={best_overall['RMSE']:.3f}"
)
plt.xlabel("Fecha")
plt.ylabel("Temperatura")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 42. Interpretación general de los experimentos

- **Ventana:** más historia puede ayudar hasta cierto punto, pero también eleva el costo y puede introducir información redundante. La elección debe justificarse con validación, no solo con intuición.
- **Horizonte:** pronosticar más lejos suele ser más difícil porque disminuye la influencia directa de la última observación y se acumula incertidumbre.
- **Variables:** la temperatura pasada constituye una señal fuerte; presión, humedad, viento, precipitación, ciclos temporales e identificación de estación pueden aportar contexto adicional.
- **Arquitectura:** aumentar profundidad o usar bidireccionalidad incrementa los parámetros. Una CNN puede extraer patrones locales antes de la GRU. Ninguna alternativa es universalmente superior: se comparan error, estabilidad y costo.
- **Persistencia:** en temperatura horaria es una referencia exigente, porque la variable cambia gradualmente. Una red compleja solo se justifica si mejora de manera consistente esa referencia.

## 43. Preguntas de reflexión resueltas

1. **¿Por qué es una serie multivariada?** Porque cada instante contiene varias variables relacionadas —temperatura, presión, humedad, viento, precipitación, tiempo y estación— y se usa su evolución conjunta.

2. **¿Qué representa cada dimensión del tensor?** En `(muestras, pasos temporales, características)`, la primera dimensión cuenta ventanas, la segunda las horas históricas y la tercera las variables observadas en cada hora.

3. **¿Por qué la división debe ser cronológica?** Para entrenar con el pasado y evaluar en datos posteriores, imitando el uso real del pronóstico.

4. **¿Qué fuga produciría mezclar los datos?** Ventanas del futuro podrían influir en el entrenamiento y compartir patrones muy cercanos con prueba, generando métricas artificialmente optimistas.

5. **¿Por qué los escaladores se ajustan solo con entrenamiento?** Porque usar medias o desviaciones de validación o prueba transferiría información futura al modelo.

6. **¿Qué representa la ventana histórica?** Es el bloque de observaciones anteriores que el modelo recibe para resumir el estado reciente de la atmósfera.

7. **¿Qué representa el horizonte?** Es la distancia temporal entre la última observación de entrada y el instante que se desea pronosticar.

8. **¿Qué ventaja aporta la presión?** Ayuda a describir cambios atmosféricos y sistemas meteorológicos que pueden anticipar variaciones de temperatura y precipitación.

9. **¿Por qué usar seno y coseno para hora y día?** Conservan la naturaleza circular: las 23:00 están cerca de las 00:00 y el final del año está cerca de su inicio.

10. **¿Qué aporta identificar la estación?** Permite aprender diferencias persistentes de altitud, clima medio y amplitud térmica entre ubicaciones.

11. **¿Qué significa persistencia?** Supone que la temperatura futura será igual a la última observada; es una línea base sencilla y fuerte para horizontes cortos.

12. **¿Por qué superar la persistencia?** Si un modelo complejo no mejora una regla sin entrenamiento, su costo adicional no queda justificado.

13. **¿Diferencia entre MAE y RMSE?** MAE promedia el tamaño absoluto de los errores; RMSE penaliza más los errores grandes al elevarlos al cuadrado antes de promediar.

14. **¿Qué significa un \(R^2\) negativo?** Que el modelo predice peor que usar siempre la media del objetivo del conjunto evaluado.

15. **¿Qué debe observarse en los residuos?** Deben distribuirse alrededor de cero sin tendencias, curvaturas ni cambios sistemáticos de dispersión. Patrones visibles indican sesgo o relaciones no aprendidas.

16. **¿Por qué varía el desempeño entre estaciones?** Cada estación tiene rangos, amplitudes y relaciones meteorológicas distintas; además, ciertos eventos pueden ser más difíciles de anticipar en una ubicación.

17. **¿Qué arquitectura logró el mejor equilibrio?** La tabla del experimento D permite responder con evidencia: se contrasta el menor RMSE con el número de parámetros y el tiempo. La conclusión automática identifica el mejor error y la opción más rápida.

18. **¿Una LSTM siempre supera a una SimpleRNN?** No. Depende del tamaño de datos, longitud de dependencias, ruido y ajuste. Para patrones cortos, una SimpleRNN puede ser suficiente y más económica.

19. **¿Qué cambia al aumentar el horizonte?** La relación con la última medición se debilita y normalmente aumentan MAE y RMSE; también puede disminuir \(R^2\).

20. **¿Cómo adaptar el notebook a datos reales?** Se validan columnas y unidades, se ordenan fechas, se tratan faltantes y atípicos sin usar información futura, se separa cronológicamente, se ajustan escaladores con entrenamiento y se repite la comparación por estación y horizonte.

## 44. Síntesis

El flujo experimental completo fue:

\[
	ext{Datos meteorológicos}
ightarrow 	ext{división cronológica}
ightarrow 	ext{normalización}
ightarrow 	ext{ventanas}
ightarrow 	ext{GRU/arquitecturas alternativas}
ightarrow 	ext{pronóstico}
ightarrow 	ext{evaluación}
\]

Los experimentos mostraron que el rendimiento depende de cuatro decisiones conectadas: cuánta historia se observa, qué tan lejos se predice, qué variables se incluyen y qué capacidad tiene la arquitectura. La comparación se fundamentó en MAE, RMSE, \(R^2\), parámetros, épocas, tiempo y la línea base de persistencia. Las cifras de las tablas se obtienen directamente del entrenamiento y no fueron escritas manualmente.

## 45. Reto final — Adaptación preparada para datos meteorológicos reales

El reto requiere un archivo real que no forma parte del cuadernillo. Para no inventar mediciones, se deja preparada la función de carga y validación. Al recibir un CSV real, se sustituye la generación sintética por esta celda y se repite el mismo flujo experimental.

El archivo debe contener: `timestamp`, `station`, `temperature`, `pressure`, `humidity`, `wind_speed` y `precipitation`. La preparación ordena cronológicamente, elimina duplicados, interpola únicamente dentro de cada estación y marca atípicos mediante el rango intercuartílico. En un proyecto real, los límites físicos y el tratamiento final deben acordarse con especialistas del dominio.

In [ ]:
REQUIRED_REAL_COLUMNS = [
    "timestamp", "station", "temperature", "pressure", "humidity",
    "wind_speed", "precipitation"
]


def load_and_prepare_real_weather(csv_path):
    real_df = pd.read_csv(csv_path)
    missing_columns = sorted(set(REQUIRED_REAL_COLUMNS) - set(real_df.columns))
    if missing_columns:
        raise ValueError(f"Faltan columnas obligatorias: {missing_columns}")

    real_df = real_df[REQUIRED_REAL_COLUMNS].copy()
    real_df["timestamp"] = pd.to_datetime(real_df["timestamp"], errors="coerce")
    real_df = real_df.dropna(subset=["timestamp", "station"])
    real_df = real_df.drop_duplicates(["station", "timestamp"])
    real_df = real_df.sort_values(["station", "timestamp"]).reset_index(drop=True)

    numeric_columns = [
        "temperature", "pressure", "humidity", "wind_speed", "precipitation"
    ]
    real_df[numeric_columns] = real_df.groupby("station")[numeric_columns].transform(
        lambda group: group.interpolate(limit_direction="both")
    )

    outlier_summary = []
    for station_name, group in real_df.groupby("station"):
        for column in numeric_columns:
            q1, q3 = group[column].quantile([0.25, 0.75])
            iqr = q3 - q1
            lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            count = int(((group[column] < lower) | (group[column] > upper)).sum())
            outlier_summary.append({
                "station": station_name,
                "variable": column,
                "possible_outliers": count,
            })

    return real_df, pd.DataFrame(outlier_summary)


print("Función preparada. Ejemplo de uso:")
print("real_weather_df, real_outliers = load_and_prepare_real_weather('meteorologia_real.csv')")
print("Después se repiten la división cronológica, las ventanas y los modelos.")

### Limitaciones documentadas

- Los resultados experimentales de este cuadernillo pertenecen a datos sintéticos y no equivalen a un pronóstico operativo.
- Un CSV real puede presentar intervalos irregulares, sensores descalibrados, cambios de estación y periodos largos sin datos.
- La interpolación no debe atravesar vacíos extensos sin una revisión previa.
- Una GRU entrenada con pocas estaciones puede no generalizar a lugares o años distintos.
- Para un uso real se requieren validación temporal en varios periodos, comparación con modelos meteorológicos especializados y seguimiento continuo del error.
- La arquitectura bidireccional solo procesa la ventana histórica ya disponible; no debe recibir observaciones posteriores al instante de pronóstico.

## Conclusión de la entrega

La **Parte XIV** quedó desarrollada con los cuatro experimentos, resultados cuantitativos, visualizaciones, interpretación, respuestas de reflexión y preparación del reto con datos reales. El notebook conserva las secciones previas para que el proceso completo pueda revisarse y reproducirse en Google Colab.